# PGC comparison: QC, clustering, and germ cell / PGCLC identification

*In vivo 6 pcw (Garcia-Alonso) and in vitro day 4 (Joao's dataset), one dataset per run.* Both can use the CELLRANGER_DIRS function.

### Setting up the environment

Either use pip or set-up a conda environment.

```bash
# install miniconda if you don't have it already

conda create -n pgc python=3.11 -y
conda activate pgc

conda install -c conda-forge -y \
    scanpy anndata pandas numpy scipy matplotlib seaborn \
    harmonypy leidenalg python-igraph ipykernel
```

Versions this was run against: scanpy 1.12.2, anndata 0.13.2, harmonypy 2.0.0,
leidenalg 0.12.0, python-igraph 1.0.0.

In [ ]:
import os

os.chdir("/storage/bioinformatics/") # CHANGE TO YOUR DIRECTORY

### Arrange the Cell Ranger files (run once)

`sc.read_10x_mtx` reads a directory that contains `matrix.mtx.gz`, `barcodes.tsv.gz` and `features.tsv.gz`. The E-MTAB-10551 download arrives as one flat folder with the sample name prefixed onto each file, so move them into one directory per sample first:

```bash
# Create folders for each sample
mkdir -p data/invivo/FCA_GND8622630 data/invivo/FCA_GND9332064 data/invivo/FCA_GND9332065

# Rename the first sample's files
mv data/FCA_GND8622630_barcodes.tsv.gz data/invivo/FCA_GND8622630/barcodes.tsv.gz
mv data/FCA_GND8622630_features.tsv.gz data/invivo/FCA_GND8622630/features.tsv.gz
mv data/FCA_GND8622630_matrix.mtx.gz   data/invivo/FCA_GND8622630/matrix.mtx.gz

# Repeat for sample 9332064
mv data/FCA_GND9332064_barcodes.tsv.gz data/invivo/FCA_GND9332064/barcodes.tsv.gz
mv data/FCA_GND9332064_features.tsv.gz data/invivo/FCA_GND9332064/features.tsv.gz
mv data/FCA_GND9332064_matrix.mtx.gz   data/invivo/FCA_GND9332064/matrix.mtx.gz

# Repeat for sample 9332065
mv data/FCA_GND9332065_barcodes.tsv.gz data/invivo/FCA_GND9332065/barcodes.tsv.gz
mv data/FCA_GND9332065_features.tsv.gz data/invivo/FCA_GND9332065/features.tsv.gz
mv data/FCA_GND9332065_matrix.mtx.gz   data/invivo/FCA_GND9332065/matrix.mtx.gz
```

## Overview

**Run this notebook once per dataset.** Set `DATASET` below to `"invivo"` or `"invitro"`, point the paths at your files, and run all cells. It is not written to run both datasets in one pass, because the loaders are dataset-specific.

Notes:

1. The in vivo matrices are **whole gonad**. Every cell type is in there, and isolating germ cells is part of the analysis (Section 5).
2. **`SOX17` alone does not mean germ cell.** Definitive endoderm is `SOX17`+ too. The call needs `SOX17`+ *and* `NANOS3`+ *and* `PRDM1`+ *and* `TFAP2C`+, with `POU5F1`/`NANOG` retained and `FOXA2` negative.

In [ ]:
import os
import numpy as np
import pandas as pd
import scipy.sparse as sp
import scanpy as sc
import anndata as ad
import matplotlib.pyplot as plt
import seaborn as sns

sc.settings.verbosity = 1
sc.settings.set_figure_params(dpi=100, dpi_save=150, frameon=False)

## Configuration

Everything you are likely to want to change lives in this one cell.

In [ ]:
# "invivo" (Garcia-Alonso, 6 pcw gonad) or "invitro" (Chen, day 4 PGCLC)
DATASET = "invivo"
OUTDIR  = "results_scanpy_joao_v3" 

# ---- in vivo input: per-sample Cell Ranger directories ----------------------
# From E-MTAB-10551 processed files. Each directory must contain
# matrix.mtx.gz, barcodes.tsv.gz and features.tsv.gz.
CELLRANGER_DIRS = {
    "sample5":   'data/Castillo-Venzor_v2/SIGAD7_d4i_s33/',
    "sample9": 'data/Castillo-Venzor_v2/SIGAD7_d4i_s34/',
    "sample10": 'data/Castillo-Venzor_v2/SIGAD7_d4i_s35/',
    "sample14": 'data/Castillo-Venzor_v2/SIGAD7_d4i_s36/',
}

# ---- in vitro input: dense count tables -------------------------------------
# Chen's GEO supplementary files are gene x cell text tables, gzip'd, not
# Cell Ranger mtx, so they load through a different reader below.
TABLE_FILES = {
    "ucla1.b1": "data/invitro/GSM4202926_ucla1.batch1.PGCLCd4.rawmatrix.txt.gz",
    "ucla1.b2": "data/invitro/GSM4202932_ucla1.batch2.PGCLCd4.rawmatrix.txt.gz",
    "ucla2.b1": "data/invitro/GSM4202944_ucla2.batch1.PGCLCd4.rawmatrix.txt.gz",
    "ucla2.b2": "data/invitro/GSM4202950_ucla2.batch2.PGCLCd4.rawmatrix.txt.gz",
}

# ---- QC thresholds ----------------------------------------------------------
# Starting points only. Look at the violin/scatter plots in Section 2.
MIN_GENES   = 200
MAX_GENES   = 20000     # doublet-ish upper tail; set from the violin plot
MAX_PCT_MT  = 20
MIN_CELLS   = 3         # a gene must appear in at least this many cells to keep it
N_HVG       = 2000
N_PCS       = 30
CLUSTER_RES = 0.3

### Marker panels

In [ ]:
PANELS = {
    "pgclc":     ["NANOS3","SOX17","TFAP2C"], #,"PRDM1","DND1","DPPA3","KIT","ALPL"
    "pluri":     ["POU5F1","NANOG","LIN28A"],
    "endoderm":  ["FOXA2","GATA6","HHEX","CXCR4"],
    "amnion":    ["TFAP2A","GABRP","ISL1","IGFBP3"],
    "mesoderm":  ["TBXT","HAND1","MIXL1","EOMES"],
    "late_germ": ["DAZL","DDX4","MAEL","SYCP3", "SOX4"],   # CHECK
    "neural_progen": ["SOX2"]
}

In [ ]:
os.makedirs(OUTDIR, exist_ok=True)

def savefig(name):
    plt.savefig(os.path.join(OUTDIR, name), dpi=150, bbox_inches="tight")
    print("wrote", os.path.join(OUTDIR, name))

## 1. Load

### Where the data comes from

The two datasets are deposited very differently, which is why there are two loaders below.

**In vivo (Garcia-Alonso 2022, *Nature* 607:540).** Processed Cell Ranger output lives on the E-MTAB-10551 record at ArrayExpress/BioStudies (`https://www.ebi.ac.uk/biostudies/arrayexpress/studies/E-MTAB-10551`, Files tab, filtered to *processed*). **Heads up: these matrices are whole gonad.** Every somatic and germ cell type is present, and there is no pre-filtered "germ cells only" matrix here; that only exists as the annotated `.h5ad` on the Reproductive Cell Atlas (see option (c) below). Isolating germ cells from this whole-gonad object is Section 5.

**In vitro (Preka et al. unpublished 2019).** 

In [ ]:
def load_cellranger(dirs):
    parts = []
    for s, d in dirs.items():
        if not os.path.isdir(d):
            raise FileNotFoundError(f"Cell Ranger dir not found: {d}")
        # var_names="gene_symbols" uses the symbol column of features.tsv, which
        # is what makes the "MT-" mitochondrial prefix work later in Section 2.
        a = sc.read_10x_mtx(d, var_names="gene_symbols", make_unique=True)
        a.obs["sample"] = s
        sc.pp.filter_cells(a, min_genes=100)
        print(f"  {s}: {a.n_obs} cells x {a.n_vars} genes")
        parts.append(a)
    if len(parts) == 1:
        return parts[0]
    return ad.concat(parts, join="outer", index_unique="-")


def load_tables(files):
    parts = []
    for s, f in files.items():
        if not os.path.exists(f):
            raise FileNotFoundError(f"table not found: {f}")
        # genes x cells is expected. If your file is cells x genes, drop the .T.
        df = pd.read_csv(f, sep="\t", index_col=0)
        a = ad.AnnData(df.T)
        a.X = sp.csr_matrix(a.X)
        a.var_names_make_unique()
        a.obs["sample"] = s
        sc.pp.filter_cells(a, min_genes=100)
        print(f"  {s}: {a.n_obs} cells x {a.n_vars} genes")
        parts.append(a)
    if len(parts) == 1:
        return parts[0]
    return ad.concat(parts, join="outer", index_unique="-")

In [ ]:
print(f"loading ({DATASET}) ...")
if DATASET == "invivo":
    adata = load_cellranger(CELLRANGER_DIRS)
elif DATASET == "invitro":
    adata = load_tables(TABLE_FILES)
else:
    raise ValueError("DATASET must be 'invivo' or 'invitro'")

sc.pp.filter_genes(adata, min_cells=MIN_CELLS)
print(f"combined: {adata.n_obs} cells x {adata.n_vars} genes")

## 2. Quality control

Mitochondrial fraction, gene count, and UMI count per cell.

In [ ]:
adata.var["mt"] = adata.var_names.str.startswith("MT-")
sc.pp.calculate_qc_metrics(adata, qc_vars=["mt"], percent_top=None,
                           log1p=False, inplace=True)

if adata.obs["pct_counts_mt"].max() == 0:
    print("WARNING: pct_counts_mt is all zero. Features may be Ensembl IDs "
          "rather than symbols; reload with var_names='gene_symbols', or match "
          "the mito prefix to your gene id namespace instead of 'MT-'.")

In [ ]:
sc.pl.violin(adata, ["n_genes_by_counts", "total_counts", "pct_counts_mt"],
             groupby="sample", stripplot=False, rotation=90, show=False)
fig = plt.gcf()
fig.set_size_inches(17, 5)          # widen the figure
fig.subplots_adjust(wspace=1)       # then apply spacing
savefig("qc_violin.png")
plt.show()

sc.pl.violin(adata, ["n_genes_by_counts", "total_counts", "pct_counts_mt"],
             groupby="sample", jitter=0.4, rotation=90, show=False)
fig = plt.gcf()
fig.set_size_inches(17, 5)
fig.subplots_adjust(wspace=1)
savefig("qc_violin_points.png")
plt.show()

In [ ]:
sc.pl.scatter(adata, x="total_counts", y="pct_counts_mt", color="sample", show=False)
savefig("qc_scatter_mt.png")
plt.show()

sc.pl.scatter(adata, x="total_counts", y="n_genes_by_counts", color="sample", show=False)
savefig("qc_scatter_genes.png")
plt.show()

**Look at these plots before trusting `MIN_GENES` / `MAX_GENES` / `MAX_PCT_MT` set here.** They are starting points for 10x scRNA-seq, not universal constants.

In [ ]:
MIN_GENES  = 500
MAX_GENES  = 8000
MAX_PCT_MT = 15

n0 = adata.n_obs
adata = adata[(adata.obs["n_genes_by_counts"] > MIN_GENES) &
              (adata.obs["n_genes_by_counts"] < MAX_GENES) &
              (adata.obs["pct_counts_mt"] < MAX_PCT_MT)].copy()
print(f"QC: {n0} -> {adata.n_obs} cells")
print(adata.obs["sample"].value_counts())

## 3. Normalise, find variable features, PCA

Standard log-normalisation and PCA.

In [ ]:
adata.layers["counts"] = adata.X.copy()
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)
sc.pp.highly_variable_genes(adata, n_top_genes=N_HVG)

sc.pp.pca(adata, n_comps=N_PCS, mask_var="highly_variable")

## 4. Integrate across samples (if needed), then cluster

Harmony corrects the PCA across samples so the same cell type from different
donors or batches lands together. Correcting when there is nothing to correct
can pull genuinely different cells on top of each other. So the
next cell measures how separated the samples actually are, and integration only
runs if that separation is large enough to matter.

### First, is integration actually needed?

For every cell, the score below counts how many of its nearest neighbours come
from its own sample, and compares that with the fraction expected if the samples
were mixed at random. It is rescaled so that **0 means perfectly mixed** and
**1 means the samples share no neighbours at all**. Rough reading:

| Score | Interpretation |
|---|---|
| below 0.1 | samples already well mixed, integration probably unnecessary |
| 0.1 to 0.2 | mild structure, judgement call, look at the plot |
| above 0.2 | samples clearly separate, integration likely worthwhile |

In [ ]:
from sklearn.neighbors import NearestNeighbors

INTEGRATE         = "auto"   # "auto", or True / False to force the decision
SEPARATION_CUTOFF = 0.20     # "auto" runs harmony above this


def batch_separation(a, batch_key="sample", use_rep="X_pca", k=30):
    """How separated are the samples in PCA space?

    Returns (observed, expected, score). `observed` is the mean fraction of each
    cell's k nearest neighbours drawn from its own sample; `expected` is that
    fraction if samples were mixed at random. `score` rescales the two so 0 is
    perfectly mixed and 1 is completely separated.
    """
    X = np.asarray(a.obsm[use_rep])
    lab = a.obs[batch_key].astype(str).to_numpy()
    k = int(min(k, a.n_obs - 1))
    idx = (NearestNeighbors(n_neighbors=k + 1).fit(X)
           .kneighbors(X, return_distance=False)[:, 1:])
    observed = float((lab[idx] == lab[:, None]).mean())
    p = pd.Series(lab).value_counts(normalize=True)
    expected = float((p ** 2).sum())
    score = (observed - expected) / (1 - expected) if expected < 1 else np.nan
    return observed, expected, score


multi = adata.obs["sample"].nunique() > 1
separation = np.nan

if multi:
    observed, expected, separation = batch_separation(adata)
    print(f"samples: {list(adata.obs['sample'].unique())}")
    print(f"same-sample neighbours: observed {observed:.2f}, "
          f"expected if mixed {expected:.2f}")
    print(f"separation score:       {separation:.2f}   "
          f"(0 = fully mixed, 1 = fully separated)")
    sc.pl.pca(adata, color=["sample"], show=False)
    savefig("pca_by_sample_preintegration.png")
    plt.show()
else:
    print("only one sample present, nothing to integrate")

**A high score is not automatically a batch effect.** Samples can separate for
reasons you would not want to remove: the in vivo donors differ in sex, and the
in vitro lines are genuinely different genetic backgrounds. Samples also separate
when they simply contain different proportions of cell types, which is real
biology rather than a technical artefact.

So treat the number as a prompt to look at the plot, not as the decision itself.
If the samples separate because one is mostly one cell type, integrating will
force those populations together and destroy the thing you are trying to see. Set
`INTEGRATE` to `True` or `False` explicitly whenever you have a reason to
override the automatic call.

In [ ]:
INTEGRATE = False

if INTEGRATE == "auto":
    do_integrate = bool(multi) and (separation > SEPARATION_CUTOFF)
    why = f"auto: separation {separation:.2f} vs cutoff {SEPARATION_CUTOFF}"
else:
    do_integrate = bool(INTEGRATE) and bool(multi)
    why = f"set manually to {INTEGRATE}"

rep = "X_pca"

if do_integrate:
    import harmonypy
    ho = harmonypy.run_harmony(adata.obsm["X_pca"], adata.obs, ["sample"])
    Z = np.asarray(ho.Z_corr)
    if Z.shape[0] != adata.n_obs:   # harmonypy 2.x is (cells, PCs); 1.x is (PCs, cells)
        Z = Z.T
    adata.obsm["X_pca_harmony"] = Z
    rep = "X_pca_harmony"
    _, _, after = batch_separation(adata, use_rep="X_pca_harmony")
    print(f"ran harmony ({why})")
    print(f"separation after harmony: {after:.2f}  (was {separation:.2f})")
else:
    print(f"skipped integration ({why}); using uncorrected PCA")

sc.pp.neighbors(adata, n_pcs=N_PCS, use_rep=rep)
sc.tl.leiden(adata, resolution=CLUSTER_RES, flavor="igraph", n_iterations=2,
             directed=False)
sc.tl.umap(adata)

In [ ]:
sc.pl.umap(adata, color=["leiden", "sample"], wspace=0.3, show=False)
savefig("umap_clusters.png")
plt.show()

## 5. Score marker panels and identify cell types

`sc.tl.score_genes` gives an average expression score per cell for each gene set, corrected against a background of similarly-expressed genes.

In [ ]:
for nm, genes in PANELS.items():
    present = [g for g in genes if g in adata.var_names]
    missing = set(genes) - set(present)
    if missing:
        print(f"  {nm}: not found -> {sorted(missing)}")
    if present:
        sc.tl.score_genes(adata, present, score_name=f"{nm}_score")

score_cols = [f"{nm}_score" for nm in PANELS if f"{nm}_score" in adata.obs]

### Per-cluster summary

Mean panel scores plus mean expression of the genes that actually decide the call, one row per cluster.

In [ ]:
key_genes = [g for g in ["NANOS3","SOX17","TFAP2C"] if g in adata.var_names] #"PRDM1",

df = sc.get.obs_df(adata, keys=[*score_cols, *key_genes, "leiden"])
summ = df.groupby("leiden", observed=True).mean().round(3)
summ.insert(0, "n_cells", df.groupby("leiden", observed=True).size())

#summ.to_csv(os.path.join(OUTDIR, "cluster_summary.csv"))
summ

In [ ]:
score_means = adata.obs.groupby("leiden", observed=True)[score_cols].mean()

plt.figure(figsize=(8, max(3, 0.4 * score_means.shape[0])))
sns.heatmap(score_means, annot=True, fmt=".2f", cmap="RdBu_r", center=0,
            cbar_kws={"label": "mean module score"})
plt.xlabel("module")
plt.ylabel("leiden cluster")
plt.title("Module scores per cluster")
savefig("module_scores_by_cluster.png")
plt.show()

In [ ]:
panels_present = {k: [g for g in v if g in adata.var_names] for k, v in PANELS.items()}
sc.pl.dotplot(adata, panels_present, groupby="leiden", show=False)
savefig("markers_dotplot.png")
plt.show()

In [ ]:
feat = [g for g in ["NANOS3", "SOX17", "TFAP2C"] if g in adata.var_names] 
sc.pl.umap(adata, color=feat, ncols=2, cmap="viridis", show=False)
savefig("markers_umap.png")
plt.show()

## Re-loading the h5ad file

In [ ]:
adata = sc.read_h5ad('/Users/u5756320/Library/CloudStorage/OneDrive-UniversityofWarwick/Documents/Mini-Project related/Mini Project 2 July-Sept 2026/Mini-Project-2/results_scanpy_joao/invitro_integrated_scanpy_joao.h5ad')

In [ ]:
sc.pl.umap(adata, color=["leiden", "sample"], wspace=0.3, show=False)
savefig("umap_clusters.png")
plt.show()

In [ ]:
adata.obs['cell_type'] = adata.obs['leiden'].astype(str)

# 2. Overwrite only the clusters you want to manually name
adata.obs.loc[adata.obs['leiden'] == '0', 'cell_type'] = 'PGCLC'
adata.obs.loc[adata.obs['leiden'] == '1', 'cell_type'] = 'PGCLC'
adata.obs.loc[adata.obs['leiden'] == '2', 'cell_type'] = 'PGCLC'
adata.obs.loc[adata.obs['leiden'] == '15', 'cell_type'] = 'PGCLC'
adata.obs.loc[adata.obs['leiden'] == '3', 'cell_type'] = 'cluster 3'
adata.obs.loc[adata.obs['leiden'] == '4', 'cell_type'] = 'cluster 4'
adata.obs.loc[adata.obs['leiden'] == '5', 'cell_type'] = 'cluster 5'
adata.obs.loc[adata.obs['leiden'] == '6', 'cell_type'] = 'cluster 6'
adata.obs.loc[adata.obs['leiden'] == '7', 'cell_type'] = 'cluster 7'
adata.obs.loc[adata.obs['leiden'] == '8', 'cell_type'] = 'cluster 8'
adata.obs.loc[adata.obs['leiden'] == '9', 'cell_type'] = 'cluster 9'
adata.obs.loc[adata.obs['leiden'] == '10', 'cell_type'] = 'cluster 10'
adata.obs.loc[adata.obs['leiden'] == '11', 'cell_type'] = 'cluster 11'
adata.obs.loc[adata.obs['leiden'] == '12', 'cell_type'] = 'cluster 12'
adata.obs.loc[adata.obs['leiden'] == '13', 'cell_type'] = 'cluster 13'
adata.obs.loc[adata.obs['leiden'] == '14', 'cell_type'] = 'cluster 14'

#adata.obs.loc[adata.obs['leiden'].isin(['5', '7']), 'cell_type'] = 'Mixed/Ambiguous'

# 3. Convert back to categorical
adata.obs['cell_type'] = adata.obs['cell_type'].astype('category')

In [ ]:
umap = adata.obsm['X_umap']
is_pgclc = (adata.obs['cell_type'] == 'PGCLC').values

fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(umap[~is_pgclc, 0], umap[~is_pgclc, 1], c='lightgrey', s=8, linewidths=0)
ax.scatter(umap[is_pgclc, 0], umap[is_pgclc, 1], c='red', s=15, linewidths=0, label='PGCLC')
ax.set_xlabel('UMAP1'); ax.set_ylabel('UMAP2')
ax.legend(frameon=False, loc='best')
ax.set_title('PGCLCs (n= {})'.format(is_pgclc.sum()))
ax.axis('off')
plt.tight_layout()
savefig("umap_PGCLC.png")
plt.show()

In [ ]:
adata.obs['cell_type'].value_counts()
print(adata.obs['sample'].value_counts())

In [ ]:
adata.obs.loc[is_pgclc, 'sample'].value_counts()

### Decide

Read the dot plot above yourself, then set the passing clusters here and subset. This is gated behind `RUN_SUBSET` so a run-all does not execute it on a placeholder.

In [ ]:
RUN_SUBSET = True            # set True once you have chosen the clusters below
PGC_CLUSTERS = ["10", "12"]    # replace with your own cluster ids from the dot plot

if RUN_SUBSET:
    germ = adata[adata.obs["leiden"].isin(PGC_CLUSTERS)].copy()
    print(germ.obs["sample"].value_counts())
    germ.write(os.path.join(OUTDIR, f"{DATASET}_germ.h5ad"))

## 6. Cluster markers and save

`sc.tl.rank_genes_groups` gives a one-vs-rest marker table per cluster, useful for confirming cluster identity.

In [ ]:
sc.tl.rank_genes_groups(adata, "leiden", method="wilcoxon")
markers = sc.get.rank_genes_groups_df(adata, group=None)
markers.to_csv(os.path.join(OUTDIR, "cluster_markers.csv"), index=False)

top5 = (markers.sort_values("logfoldchanges", ascending=False)
               .groupby("group", observed=True)
               .head(5)
               .sort_values("group"))
top5

In [ ]:
adata.write(os.path.join(OUTDIR, "invitro_integrated_scanpy_joao.h5ad"))
print("wrote", os.path.join(OUTDIR, "invitro_integrated_scanpy_joao.h5ad"))